# TEMA 4: APLICACIONES PRÁCTICAS DE VISIÓN POR COMPUTADOR

Entrenar un modelo convolucional de cero es ineficiente en muchas ocasiones. Si tienes pocos datos puede haber sobreajuste. Si tienes poco recurso computacional no puedes entrenarla. El transfer learning es una técnica que permite reutilizar modelos ya entrenados en tareas similares, resolviendo así el problema de clasificación con una inversión menor.

## ¿Por qué el transfer learning tiene sentido?

Parte de una observación clave sobre las redes convolucionales. Las primeras capas de la misma aprenden patrones generales como bordes, texturas, esquinas, formas simples y las capas mas profundas van capturando rasgos específicos de los datos de entranmiento. Eso significa que las primeras capas que son generales pueden ser útiles para otro tipo de tareas visuales aunque el conjunto de clases sea distinto. 

## Forma más tipica de transfer learning

Utilizas una CNN ya entrenada, por ejemplo, ImageNet y eliminas su última capa de clasificación sustituyéndola por otra nueva. Ya vimos en la parte de la estructura de CNN como despues de las capas de convolución y Flatten llega el momento de las capas ocultas y la parte de clasificación con función de activación softmax para extraer probabilidad de pertenecer a una clase. 

Hay dos enfoques, permitir que algunas capas se ajusten y hacer fine-tuning o congelar las capas previas, feature extraction- 

## Ventajas del transfer learning

- Reducir el tiempo de entrenamiento-> solo afina detalles en lugar de inicializar de manera aleatoria, por tanto menos epocas, menos riesgo de atascarse en mínimos locales 
- Se pueden emplear configuraciones mas ligeras en consecuencia de esto
- Rapidez, adaptabilidad y eficiencia

## Principales arquitecturas pre-entrenadas

Tenemos decenas de arquitecturas, vamos a ver las más importantes y además, como cada una de ellas surge como solución a problemas muy concretas tiene unas características muy concretas.  Disponibles en keras.applications. 

> En muchos proyectos, se parten de estas redes entrenadas con ImageNet y se adaptan con Transfer Learning


### VGG16/VGG19 

Es la estructura simple con capas convolucionales 3 por 3 apiladas una tras otra. NO HAY TRUCOS ESPECIALES. Son fáciles de modificar porque son las predecibles

Eso sí, son muy pesadas , con muchos parámetros porque son capas tras capas con nada de optimización de eficiencia. 

Se emplean como recursos didácticos

### ResNet

Son bloques residuales (res) es decir hay skip connection. Lo que hacen estos skip connection son evitar el desvanecimiento de gradiente. Cuando entra una entrada a una capa, esta no se transforma completamente. Lo que hace es sumarle la entrada sin modificar F(x) + x -> por tanto el gradiente no se diluye por muchas capas que pongas y por muy profunda que sea.

Innevitablemente se vuelve más compleja de interpretar.

Se emplea en aplicaciones genéricas, clasificación más precisa


### Inception

Combina varios filtros en paralelo de distinto tamaño dentro del mismo bloque, luego concatena esos resultados. Como no sabes si el patron en la imagen es grande o pequeño, miras varias escalas a la vez. Se dice que es muy eficiente ya que se reduce dimensionalidad antes de poner convoluciones más caras con convoluciones 1 por 1

Es dificil de personalizar 

Se emplea cuando quieres escenarios mixtos de precisión y eficiencia

### EfficientNet

Hace compound scaling, es decir optimización conjunta de profundidad, anchura (mas filtros por cada cada) y resolución de entrada. 

Hay que ajustar bien el tamaño de entrada a la red 

Cuando tienes dispositivos con recursos limitados, producción ligera

### Vision Transformers (ViT)

No hay convoluciones. Divide la imagen en parches y trata cada parche de la imagen como un token, igual que una palabra en un NLP. Utiliza self-attention de los Transformers para tener relaciones globales desde el principio ya que cada parche mira a los demás independientemente de la distancia a consecuencia del tamaño de la imagen y la partición por parches. En una CNN con convoluciones se require pasar por capas para que la información llegue de una capa a otra. 

No obstante, se necesitan grandes dataset o preentrenamiento muy cuidado, ya no nos aprovechamos del sesgo útil que tienen las CNN ya que ahora tiene que aprenderlo desde los datos al utilizar parches y self attention.TIENE QUE DESCUBRIR LA LOCALIDAD POR SU CUENTA.  En las CNN el sesgo inductivo era que el parte suponía que era importante lo local siempre: CONVOLUCIONES LOCALES-> ahora empieza de cero y tiene que aprenderlo por sí mismo.

Tareas de clasificación con relaciones espaciales complejas


> No hay una solución universal: el modelo óptimo depende del contexto. Conocer estas diferencias es lo que permite tomar decisiones técnicas fundamentadas, no solo reproducir ejemplos. Adoptar una arquitectura preentrenada no es solo una cuestión de conveniencia. Es una forma estratégica de aprovechar años de desarrollo acumulado, integrando en pocos minutos modelos que han sido optimizados, validados y puestos a prueba en entornos reales por cientos de investigadores.

In [ ]:
# ejemplos

## FEATURE EXTRACTION Y FINE-TUNING

- Feature extraction: congelas todas las capas convolucionales del modelo preentrenado; solo entrenas una "cabeza" nueva (capas Dense) encima. La red preentrenada actúa como extractor fijo de características. Se puede sin GPU

- Fine-tuning: descongelas parte de las capas convolucionales (normalmente las últimas) y las dejas reentrenar junto con la cabeza nueva. Mayor capacidad de cómputo.

El feature extraction suele ser preferible en contextos de escasez de datos(ejemplo prototipo temprano interesa solo extraer características),
mientras que el fine-tuning ofrece mejores resultados cuando se
busca máxima precisión en un dominio específico.(etapas más posteriores con más datos)

> Ambas estrategias asumen lo mismo: las primeras capas de una CNN aprenden cosas muy genéricas (bordes, texturas, formas simples — como los detectores tipo Sobel que vimos en 3.3), útiles en cualquier dominio de imágenes. Las últimas capas son las que se especializan en las clases concretas del dataset original (ImageNet: razas de perro, tipos de coche...). Por eso en fine-tuning se descongelan típicamente las últimas capas — son las que necesitan "reaprender" para tus clases nuevas, mientras que las primeras (detectores de bordes genéricos) siguen siendo útiles tal cual.

"entrenar primero la cabeza de clasificación con las capas congeladas y, una vez estabilizada, reactivar progresivamente las últimas capas convolucionales"

Al principio, tu cabeza nueva tiene pesos aleatorios. Si desde el primer momento permites que el gradiente fluya también hacia las capas preentrenadas, esos gradientes (grandes y caóticos, porque vienen de una cabeza sin entrenar) pueden "destrozar" representaciones preentrenadas que eran buenas. Por eso primero se estabiliza la cabeza (con todo lo demás congelado), y solo después se descongela — para que los gradientes que lleguen a esas capas preentrenadas sean ya razonables Y NO DESTRUYAS LAS CAPAS CONVOLUCIONALES.

# Congelar todas las capas
base_model.trainable = False

# Descongelar las últimas 50 capas de ResNet
for layer in base_model.layers[-50:]:
layer.trainable = True
# Compilar de nuevo con menor tasa de aprendizaje
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5) MUY BAJO,
loss=’categorical_crossentropy’,
metrics=[‘accuracy’])
model.fit(train_generator,
validation_data=val_generator,
epochs=5)
